In [11]:
#----------------------------------
#Work arround code for printing latex in VSCode. If you are running directly in JupiterLab, comment the following lines and run %display latex
from latex_disp_VSCode import slatex
#----------------------------------


#Function to define n variables named x1, ... xn
def Xvar(n : int):
    X=[]
    for i in range(n):
        X.append('x'+str(i+1))
    return X

#Function to define n variables named T1, ... Tn
def Tvar(n : int):
    T=[]
    for i in range(n):
        T.append('T'+str(i+1))
    return T


#Function to define a supercommutative algebra with n even and l odd generators, whose name is given by list of strings in *lambda_lenghts, *mu_invariants.

def superalgebra(n : int,l : int, lambda_lenghts: list[[str]] = None, mu_invariants: list[[str]] = None) : 
    if lambda_lenghts is None : 
        P = PolynomialRing(QQ, n, Xvar(n)) ## n is for lamba lenghts
        P.inject_variables(verbose=False)
        P = FractionField(P)
    else : 
        if n != len(lambda_lenghts) : 
            print('The number of generators of the even part should match the length your names vector')
            return
        else: 
            P = PolynomialRing(QQ, n, lambda_lenghts)
            P.inject_variables(verbose=False)
            P = FractionField(P)
    #We will work with the exterior algebra over the odd variables
    if mu_invariants is None : 
        F = ExteriorAlgebra(P, names = Tvar(l)) ## l is for the mu invariants 
        F.inject_variables(verbose=False)
    else : 
        if l != len(mu_invariants) : 
            print('The number of generators of the odd part should match the length your names vector')
            return
        else: 
            F = ExteriorAlgebra(P, names = mu_invariants)
            F.inject_variables(verbose=False)
    return F


#Workaround for buggy ExteriorAlgebra implementations. This allows you to multiply matrices with elements over an exterior algebra. 
def mat_mult(A, B):
    n = A.nrows()
    result = [[sum(A[i,k] * B[k,j] for k in range(n)) for j in range(n)] for i in range(n)]
    return MM(result)
def mat_prod(*args):
    result = args[0]
    for m in args[1:]:
        result = mat_mult(result, m)
    return result

In [16]:
## EXAMPLE: Superalgebra of Figure 4, 2 odd variables \sigma (S) and \theta (T) and even variables a and b. 
AA = superalgebra(2,2, ['a','b'], ['T','S'])
a*b*T*S + a*b*S*T #Should give 0

0

In [18]:
#The code does not support square roots. In the defintion of the holonomy matrix below, insert the variable as you would in paper, their definition of these matrices in this notebook accounts for this. For the definition of these supermatrices, see Definition 2.2 in https://arxiv.org/pdf/2208.13664. `EI` and `AI`` stand for the inverse of matrices E and I. R is tha mtrix $\rho$. 

#First, define the 3x3 matrix space over your superalgebra A
MM = MatrixSpace(AA, 3, sparse=False)

def E(param):
    param=AA(param)
    return MM([[0,-param^2,0],[1/param^2,0,0],[0,0,1]])

def A(h, angle):
    h = AA(h)
    angle = AA(angle)
    return MM([[1,0,0], [h^2, 1, -h*angle], [h*angle, 0, 1]])

def EI(param):
    param = AA(param)
    return MM([[0,param^2,0],[-1/param^2,0,0],[0,0,1]])

def AI(h, angle):
    h = AA(h)
    angle = AA(angle)
    return MM([[1,0,0], [-h^2, 1, h*angle], [-h*angle, 0, 1]])

R = MM([[-1,0,0],[0,-1,0],[0,0,1]])

In [ ]:
#Example:
#For the peripheral arcs $\lambda_2^x$ from Figure 7, we have that
H1x = mat_prod(E(1), A(a/b, -S), A(1/(a*b), T), A(b/a, -S) ,E(1))
slatex(H1x)
#Whose asscoaited diamond is shaded in gray in Figure 8. Recall that the variables a, b are actually \sqrt(a) and \sqrt(b) from the printed version (see README.md)

<IPython.core.display.Math object>

In [21]:
#Example:
#For the peripheral arcs $\lambda_2^x$ from Figure 7, we have that
H2x = mat_prod(E(1), A(a/b, S), A(1/(a*b), -T), R, EI(a), AI(1/(a*b), -S),R, E(b),A(1/(a*b), T), A(b/a, -S) ,E(1))
slatex(H2x)

<IPython.core.display.Math object>

In [26]:
# Note that the (1,) and (2,2) entries of the matrix H2x coincide, and that these are minus the entrie (1,2) of the matrix H1x, this is the lambda expansion associated to $\lambda_1^x$. The entry (1,2) of H2x is the lambda expansion of $\lambda_2^x$ denoted by $\lambda_2$ in Figure 8. 
(H2x[0,0] == H2x[1,1], H2x[0,0] == -H1x[0,1] )

(True, True)

In [29]:
# Mu-expansions associated to the region bounded by $\lambda_1_x$ apear both in entries (3,1) and (2,3) of matrix H2x. Comaring with Theorem 2.7, we see that they shuld be he negative of each other, which is not the case. This says that the choice of flips used to compute them differ, so when gluing the superfrieze one must account for this. 
(H2x[2,0] == -H2x[1,2], -H2x[1,2] == H1x[0,2])

(False, True)

In [49]:
#The following is a simple loop computing all holonomy matrices up to Hnx. Before using it, please compile all previous cells 
def holonomy_kronecker(n : int, all_hol=False) :
    """ Computes the Holonomy matrix of the arc lambda_n^x for the Kroneckek (Annulus A(1,1))
        n - Integer corresponding to the arc $lambda_n^x$
        all_hol - If True, gives all the holonom matrices of lambda_i^x up to n, default set to False
    """
    B = mat_prod(A(1/(a*b), T), A(b/a, -S) ,E(1))
    H = B
    if all_hol == True:
        HH=[]
    if n == 1:
        B = mat_prod(E(1), A(a/b, -S), A(1/(a*b), T), A(b/a, -S) ,E(1))
        if all_hol == True:
            HH.append(B)
            return HH
        else : 
            return B
        #N = mat_prod( A(1/(a*b), ((-1) ** (i // n +1))*T), R, EI(a), AI(1/(a*b), ((-1) ** (i // n +1))S),R, E(b))
        #P = 
    else :
        if all_hol == True:
            HH.append(mat_prod(E(1), A(a/b, -S), A(1/(a*b), T), A(b/a, -S) ,E(1)))
        for i in range(n-1):
            B = mat_prod( A(1/(a*b), ((-1) ** (i%2 +1))*T), R, EI(a), AI(1/(a*b), ((-1) ** (i%2 +1))*S),R, E(b) ,B)
            if all_hol == True:
                HH.append(mat_prod(E(1), A(a/b, ((-1) ** (i%2))*S), B))
        B = mat_prod(E(1), A(a/b, ((-1) ** ((n)%2))*S), B)
        if all_hol == True :
            return HH
        else : 
            return B

In [ ]:
#Example : Third diamond, notice that the phenomena we described for he first and second diamond is reproduced for the second and third. 
slatex(holonomy_kronecker(3, True))

<IPython.core.display.Math object>